In [2]:
import pandas as pd
import missingno as msno

In [3]:
data = {
    "CustomerID": ["C001", "C002", "C003", "C004", "C004", "C005", "C006", "C007", "C008", "C008"],
    "Name": ["John", "Alice", "BOB", "BOBY", "Eve", "eve", "Steve", "Ramu", "mary", "Bob"],
    "Age": [25, 34, 17, 29, 29, 120, -5, None, 220, 30],
    "JoinDate": ["12/1/2024", "11/15/2023", "6/1/2022", "6/1/2022", "12/5/2024", "invalid_date", None, "1/1/2024", "3/5/2023", "3/5/2023"],
    "MonthlyCharges": [29.85, 56.95, 4000, 75.5, 75.5, 45.99, 60, 49.99, -30, 55],
    "Churn": ["No", "Yes", "No", "No", "No", "Yes", "No", None, "Yes", "No"]
}
df = pd.DataFrame(data)

In [4]:
df.head(10)

,CustomerID,Name,Age,JoinDate,MonthlyCharges,Churn
0,C001,John,25.0,12/1/2024,29.85,No
1,C002,Alice,34.0,11/15/2023,56.95,Yes
2,C003,BOB,17.0,6/1/2022,4000.00,No
3,C004,BOBY,29.0,6/1/2022,75.50,No
4,C004,Eve,29.0,12/5/2024,75.50,No
5,C005,eve,120.0,invalid_date,45.99,Yes
6,C006,Steve,-5.0,None,60.00,No
7,C007,Ramu,NaN,1/1/2024,49.99,None
8,C008,mary,220.0,3/5/2023,-30.00,Yes
9,C008,Bob,30.0,3/5/2023,55.00,No


In [5]:
df.describe()

,Age,MonthlyCharges
count,9.000000,10.000000
mean,55.444444,441.878000
std,70.542737,1250.560418
min,-5.000000,-30.000000
25%,25.000000,46.990000
50%,29.000000,55.975000
75%,34.000000,71.625000
max,220.000000,4000.000000


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      10 non-null     object 
 1   Name            10 non-null     object 
 2   Age             9 non-null      float64
 3   JoinDate        9 non-null      object 
 4   MonthlyCharges  10 non-null     float64
 5   Churn           9 non-null      object 
dtypes: float64(2), object(4)
memory usage: 612.0+ bytes


In [8]:
df.isnull().any()

CustomerID        False
Name              False
Age                True
JoinDate           True
MonthlyCharges    False
Churn              True
dtype: bool

In [10]:
df.columns

Index(['CustomerID', 'Name', 'Age', 'JoinDate', 'MonthlyCharges', 'Churn'], dtype='object')

In [21]:
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['JoinDate'] = df['JoinDate'].fillna(df['JoinDate'].mode()[0])
df['Churn'] = df['Churn'].fillna(df['Churn'].mode()[0])
df['Name'] = df['Name'].fillna(df['Name'].mode()[0])

In [22]:
import numpy as np

In [50]:
def replace_negative_age_with_mean(df):
    df = df.copy()
    mean_age = df.loc[(df["Age"] >= 0) & (df["Age"] <= 120), "Age"].mean()
    mean_charges = df.loc[df["MonthlyCharges"] >= 0, "MonthlyCharges"].mean()
    df["Age"] = df["Age"].apply(
        lambda x : mean_age if x < 0 else x
    )
    df["MonthlyCharges"] = df["MonthlyCharges"].apply(
        lambda x : mean_charges if x < 0 else x
    )
    return df

In [51]:
def replace_invalid_date(df):
    df = df.copy()
    mode_date = df["JoinDate"].mode()[0]
    df["JoinDate"] = df["JoinDate"].replace("invalid_date", mode_date)
    return df


In [57]:
df = replace_negative_age_with_mean(df)
df = replace_invalid_date(df)
df

,CustomerID,Name,Age,JoinDate,MonthlyCharges,Churn
0,C001,John,25.000000,12/1/2024,29.850000,No
1,C002,Alice,34.000000,11/15/2023,56.950000,Yes
2,C003,BOB,17.000000,6/1/2022,4000.000000,No
3,C004,BOBY,29.000000,6/1/2022,75.500000,No
4,C004,Eve,29.000000,12/5/2024,75.500000,No
5,C005,eve,120.000000,3/5/2023,45.990000,Yes
6,C006,Steve,42.430556,3/5/2023,60.000000,No
7,C007,Ramu,55.444444,1/1/2024,49.990000,No
8,C008,mary,220.000000,3/5/2023,494.308889,Yes
9,C008,Bob,30.000000,3/5/2023,55.000000,No


In [59]:
unique_names = df['Name'].unique()
unique_names

array(['John', 'Alice', 'BOB', 'BOBY', 'Eve', 'eve', 'Steve', 'Ramu',
       'mary', 'Bob'], dtype=object)

In [62]:
duplicated_rows = df[df.duplicated(subset='CustomerID',keep=False)]
duplicated_rows

,CustomerID,Name,Age,JoinDate,MonthlyCharges,Churn
3,C004,BOBY,29.0,6/1/2022,75.500000,No
4,C004,Eve,29.0,12/5/2024,75.500000,No
8,C008,mary,220.0,3/5/2023,494.308889,Yes
9,C008,Bob,30.0,3/5/2023,55.000000,No


In [63]:
df_clean = df.drop_duplicates(subset='CustomerID',keep='first')
df_clean

,CustomerID,Name,Age,JoinDate,MonthlyCharges,Churn
0,C001,John,25.000000,12/1/2024,29.850000,No
1,C002,Alice,34.000000,11/15/2023,56.950000,Yes
2,C003,BOB,17.000000,6/1/2022,4000.000000,No
3,C004,BOBY,29.000000,6/1/2022,75.500000,No
5,C005,eve,120.000000,3/5/2023,45.990000,Yes
6,C006,Steve,42.430556,3/5/2023,60.000000,No
7,C007,Ramu,55.444444,1/1/2024,49.990000,No
8,C008,mary,220.000000,3/5/2023,494.308889,Yes


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      10 non-null     object 
 1   Name            10 non-null     object 
 2   Age             10 non-null     float64
 3   JoinDate        10 non-null     object 
 4   MonthlyCharges  10 non-null     float64
 5   Churn           10 non-null     object 
dtypes: float64(2), object(4)
memory usage: 612.0+ bytes


In [65]:
df.isna().any()

CustomerID        False
Name              False
Age               False
JoinDate          False
MonthlyCharges    False
Churn             False
dtype: bool